# Agentic RAG sobre Markdown (Bedrock + Qdrant)

Este notebook debe estar en la misma carpeta que `rag_common.py`, `rag_save.py`, `rag_retrieve.py` y `agentic_rag.py`.

1. Instalar dependencias
2. Indexar una carpeta de ficheros `.md`
3. Probar la recuperación directamente
4. Hacer preguntas al agente

## 1. Dependencias

In [1]:
%pip install -q -U boto3 langchain-aws langchain-core qdrant-client

Note: you may need to restart the kernel to use updated packages.


## 2. Configuración

Qdrant en modo local solo admite **un cliente abierto por carpeta**, así que se crea uno aquí y se comparte con todo lo demás. Si reinicias el kernel, vuelve a ejecutar esta celda.

In [2]:
%load_ext autoreload
%autoreload 2

from IPython.display import Markdown, display

from rag_common import COLLECTION, QDRANT_PATH, REGION, LLM_MODEL, bedrock_client, open_qdrant
from rag_save import RagSave, chunk_markdown
from rag_retrieve import RagRetrieve
from agentic_rag import AgenticRAG

DOCS_DIR = "./docs"   # carpeta con tus ficheros Markdown

qdrant = open_qdrant()
bedrock = bedrock_client()
print(f"Región: {REGION} | LLM: {LLM_MODEL} | Qdrant: {QDRANT_PATH} | Colección: {COLLECTION}")

Región: eu-south-2 | LLM: eu.anthropic.claude-sonnet-5 | Qdrant: ./agentic_rag_qdrant_db | Colección: rag_md


## 3. Indexar los Markdown

Ejecuta esta celda la primera vez y cada vez que cambien los documentos. Reindexar un documento sustituye sus fragmentos anteriores; `saver.reset()` borra la colección entera.

El **tema** de cada documento se toma, por este orden, de: `topic:` o `tema:` en el front matter, la subcarpeta de primer nivel, o el título (primer `#`).

In [3]:
DOCS_DIR

'./docs'

In [4]:
#saver = RagSave(qdrant=qdrant, bedrock=bedrock)
## saver.reset()   # descomenta para empezar de cero
#stats = saver.save_directory(DOCS_DIR)
#print(f"\n{sum(stats.values())} fragmentos de {len(stats)} documentos")

Opcional: ver cómo se trocea un documento concreto sin indexarlo.

In [5]:
#from pathlib import Path
#
#ejemplo = next(Path(DOCS_DIR).rglob("*.md"))
#for c in chunk_markdown(ejemplo.relative_to(DOCS_DIR).as_posix(), ejemplo.read_text(encoding="utf-8")):
#    print(f"{c.chunk_id:<30} nivel {c.level} | {c.path or '(sin cabecera)'} | {len(c.text)} caracteres")

## 4. Recuperación directa (sin agente)

In [6]:
retriever = RagRetrieve(qdrant=qdrant, bedrock=bedrock)

# Temas y documentos indexados
for topic, docs in retriever.catalog().items():
    print(f"Tema: {topic}")
    for doc, info in docs.items():
        print(f"   {doc} — {info['title']} ({info['chunks']} fragmentos)")

Tema: .ipynb_checkpoints
   .ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md — Objetivo (77 fragmentos)
Tema: Objetivo
   ProcesoAnaliticaClientesEUROPA_V3.md — Objetivo (77 fragmentos)
   ProcesoAnaliticaClientesLATAM_V1.md — Objetivo (79 fragmentos)
Tema: ProcesoAnaliticaClientes_V2
   ProcesoAnaliticaClientes_V2.md — ProcesoAnaliticaClientes_V2 (16 fragmentos)


In [7]:
# Búsqueda semántica; topic= y doc= son filtros opcionales
for hit in retriever.search("escribe aquí tu consulta", k=5, topic=None, doc=None):
    print(f"{hit.score:.2f}  [{hit.chunk_id}]  {hit.path}")
    print("     ", hit.text[:200].replace("\n", " "), "\n")

0.21  [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#62]  Publicación y cesiones
      Una vez calculada la Franquicia de Distribución se publica y se cede a diferentes destinatarios:  ▰ Publicación en la Intranet  La publicación de la analítica se lleva a cabo por el equipo de “Franchi 

0.21  [ProcesoAnaliticaClientesEUROPA_V3.md#62]  Publicación y cesiones
      Una vez calculada la Franquicia de Distribución se publica y se cede a diferentes destinatarios:  ▰ Publicación en la Intranet  La publicación de la analítica se lleva a cabo por el equipo de “Franchi 

0.20  [ProcesoAnaliticaClientesLATAM_V1.md#61]  Publicación y cesiones
      Una vez calculada la Franquicia de Distribución se publica y se cede a diferentes destinatarios:  ▰ Publicación en la Intranet  La publicación de los resultados se lleva a cabo a través del Front, acc 

0.20  [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#44]  Fases del Proceso
      <p>Para insertar las <str

In [8]:
# Índice de cabeceras de un documento
doc = next(iter(next(iter(retriever.catalog().values()))))
for depth, path, chunk_id in retriever.outline(doc):
    print("  " * (depth - 1) + "- " + path.split(" > ")[-1], f"[{chunk_id}]")

- Objetivo [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#1]
- Alcance [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#2]
- Fases del Proceso [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#3]
- Publicación y cesiones [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#62]
- Actualización de la tabla anual [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#63]
- Control de Cambios [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#64]
- Anexo 1: Glosario de tablas y vistas utilizadas [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#65]
- Anexo 2: Glosario de términos [.ipynb_checkpoints/ProcesoAnaliticaClientesEUROPA_V3-checkpoint.md#73]


## 5. Preguntar al agente

`preguntar()` muestra la respuesta, las fuentes citadas y los pasos que ha dado el agente. Con `seguir=True` la pregunta continúa la conversación anterior.

In [9]:
rag = AgenticRAG(retriever, max_steps=10, verbose=False)
ultima = None

def preguntar(pregunta, seguir=False, ver_pasos=True):
    global ultima
    history = ultima.messages if (seguir and ultima) else None
    ultima = rag.ask(pregunta, history=history)
    display(Markdown(ultima.text))
    if ultima.sources:
        display(Markdown("**Fuentes:** " + ", ".join(f"`{s}`" for s in ultima.sources)))
    if ver_pasos:
        print(f"\n{ultima.steps} pasos:")
        for t in ultima.trace:
            print(f"  [{t['step']}] {t['tool']}({t['input']})")
    return ultima

eu.anthropic.claude-sonnet-5


In [10]:
preguntar("Explícame qué significa Franquicia de Distribución y qué mide");

Con esta información, aquí está la explicación completa:

## Franquicia de Distribución (FD)

**Qué es**

La Franquicia de Distribución es el **margen neto comercial** generado por una operación con un cliente final en el momento de su contratación. Según el glosario, es la "unidad mínima de resultados de clientes generada por la contratación de un producto de Global Markets" [ProcesoAnaliticaClientesEUROPA_V3.md#73].

Su propósito es **aislar el componente puramente comercial** del resultado de una operación, separándolo de los ingresos derivados de la gestión de riesgos o del precio de mercado [ProcesoAnaliticaClientes_V2.md#2].

**Definición técnica**

Se define como el **margen neto libre de costes financieros**, atribuible a la originación de la operación con el cliente final. Características clave [ProcesoAnaliticaClientes_V2.md#2]:
- Se calcula en el momento de la contratación y se asigna por operación.
- **Solo las operaciones con cliente final generan franquicia**; las operaciones internas, back-to-back o entre entidades del Grupo no generan franquicia.
- A diferencia de las operaciones interbancarias (que reflejan exclusivamente el precio de mercado, gestionadas por mesas de riesgo), las operaciones con cliente incluyen este margen comercial adicional.

**Qué mide**

La franquicia mide específicamente el **ingreso comercial derivado del valor añadido por ventas** (la labor de distribución/originación), diferenciándolo del resultado por gestión activa de riesgo. Esto se formaliza en la ecuación estructural que descompone el resultado total de cada mesa [ProcesoAnaliticaClientes_V2.md#2]:

> **Apalancamiento = Resultado total de la mesa – Franquicia**

- *Franquicia* = ingreso comercial por valor añadido en ventas.
- *Apalancamiento* = resultado de la gestión activa de riesgo por parte de trading.

En la práctica, el *mark-up* comercial que el equipo de ventas añade al *fair price* definido por el trader (más los ajustes XVA) es precisamente lo que se denomina franquicia [ProcesoAnaliticaClientes_V2.md#5]. La fórmula del precio final es: *Precio final = Fair Price ± Ajustes XVA + Margen comercial* [ProcesoAnaliticaClientes_V2.md#3].

**Para qué se usa**

La franquicia es una referencia clave en procesos de gestión como [ProcesoAnaliticaClientes_V2.md#8]:
- **Presupuestación**, definida por geografía, producto y mesa.
- **Incentivos comerciales**: indicador principal en los scorecards de ventas (asignación por tramos de rendimiento).
- **Modelos de reparto interno**: cross-selling, esquema XB (reparto entre geografías en operaciones transfronterizas) y sales points (incentivo en productos flow).

La herramienta de "Analítica de Clientes" administra y explota esta información operación a operación, permitiendo analizarla desde múltiples perspectivas (geografía, equipo de distribución, mesa de riesgo, gestor, cliente, canal) [ProcesoAnaliticaClientes_V2.md#0].

**Fuentes:** `ProcesoAnaliticaClientesEUROPA_V3.md#73`, `ProcesoAnaliticaClientes_V2.md#2`, `ProcesoAnaliticaClientes_V2.md#5`, `ProcesoAnaliticaClientes_V2.md#3`, `ProcesoAnaliticaClientes_V2.md#8`, `ProcesoAnaliticaClientes_V2.md#0`


6 pasos:
  [1] search({'query': 'Franquicia de Distribución qué mide'})
  [2] search({'query': 'Franquicia de Distribución objetivo alcance definición', 'doc': 'ProcesoAnaliticaClientesEUROPA_V3.md'})
  [2] read_section({'doc': 'ProcesoAnaliticaClientes_V2.md', 'heading': 'Franquicia de Distribución'})
  [3] document_outline({'doc': 'ProcesoAnaliticaClientes_V2.md'})
  [4] document_outline({'doc': 'ProcesoAnaliticaClientes_V2.md'})
  [5] read_context({'chunk_id': 'ProcesoAnaliticaClientes_V2.md#2', 'window': 3})


In [11]:
# Pregunta de seguimiento sobre la respuesta anterior
preguntar("¿Puedes dar más detalle?", seguir=True);

Aquí tienes el detalle ampliado, organizado por los apartados del documento base:

## 3.1 Modelo de la franquicia — principios de asignación

El proceso de asignación de franquicias sigue estos principios [ProcesoAnaliticaClientes_V2.md#2, #3]:
- Las franquicias generadas **automáticamente** se capturan al cierre de la jornada.
- Las **no capturadas automáticamente** deben ser solicitadas por ventas y validadas por trading.
- Las operaciones **sin cliente asignado** no generan franquicia; su resultado se asigna íntegramente como apalancamiento.
- La información se gestiona de forma **centralizada por Finanzas CIB Holding**, y cualquier modificación en la franquicia se refleja automáticamente en el cálculo del apalancamiento.

## 3.2 Componentes del pricing

El precio final de una operación con cliente se construye así [ProcesoAnaliticaClientes_V2.md#3]:

> **Precio final = Fair Price ± Ajustes XVA + Margen comercial**

El cálculo puede obtenerse de dos formas:
- **Automáticamente**, mediante las plataformas **Murex, STAR y Calypso**, por captura directa de precios o diferencias registradas ("triadas").
- **Manual supervisado**, cuando la operación no reside en plataforma (p. ej. comisiones de DCM), con validación cruzada entre ventas y trading.

(Existe documentación de referencia adicional: *"Front Office Pricing Methodology 2021 March.pdf"*).

## 3.3 Proceso de obtención del valor (flujo paso a paso)

Una vez formalizada la operación, el cálculo y registro de la franquicia sigue estas etapas [ProcesoAnaliticaClientes_V2.md#4, #5, #6, #7]:

| Paso | Descripción |
|---|---|
| 01 | El **trader** define el *fair price* y proporciona los ajustes XVA de referencia. |
| 02 | El **equipo de ventas** establece el precio final al cliente añadiendo el *mark-up* comercial. **Ese incremento es la franquicia.** |
| 03 | Se intercambia el dato de la franquicia entre vendedor y mesa; se captura en Murex o automáticamente en STAR, el mismo día de contratación (salvo excepciones por complejidad, al día siguiente). |
| 04 | La información se **consolida en proceso batch** a fin de día en la base de datos de analítica de clientes, con todos los registros de la operación (franquicia, equipo de ventas, mesa de riesgo, etc.). |
| 05 | Cualquier **modificación de la franquicia ajusta automáticamente el apalancamiento**: el resultado total del libro no cambia; un incremento de franquicia implica reducción del apalancamiento y viceversa (control clave del modelo). |
| 06 | El dato es base del **modelo de medición de performance**: se presupuesta por separado el resultado de gestión de riesgo (apalancamiento) y el de gestión comercial (franquicia); seguimiento diario/mensual en Intranet GM y Finanzas Holding, y en scorecards individuales de gestores. |
| 07 | Forma parte de **KPIs del área y transversales a nivel Grupo**, aportando el valor comercial de cada operación y sirviendo para determinar la **rentabilidad de clientes y sectores**. |

## Herramienta de cálculo ("Analítica de Clientes")

Centraliza registro, mantenimiento y análisis de eventos de franquicia [ProcesoAnaliticaClientes_V2.md#7, #8]:
- **Integración directa** con Murex, STAR y Calypso.
- **Registro por operación**, permitiendo múltiples eventos por transacción.
- **Gestión exclusiva** por Finanzas CIB Holding.
- **Trazabilidad total**: cada evento con marca de tiempo, usuario responsable y naturaleza del cambio.
- **Capacidad histórica** y control de cambios sin afectar el resultado total consolidado.

## Usos internos de la franquicia

- **Presupuestación**: por geografía, producto y mesa, separada del apalancamiento.
- **Incentivos comerciales**: indicador principal en scorecards de ventas, por tramos de rendimiento.
- **Modelos de reparto interno**:
  - *Cross-selling*: reconocimiento en ventas a clientes de Banca Comercial o Empresas.
  - *Esquema XB*: reparto entre geografías en operaciones transfronterizas.
  - *Sales points*: incentivo específico para productos flow.

Si necesitas, puedo profundizar también en los **"Acuerdos de negocio calculados en la analítica"** (sección 4 del mismo documento) o en cómo se aplica este modelo concretamente en los procesos técnicos de **Europa** o **LATAM**.

**Fuentes:** `ProcesoAnaliticaClientes_V2.md#3`


2 pasos:
  [1] read_context({'chunk_id': 'ProcesoAnaliticaClientes_V2.md#5', 'window': 3})
  [1] search({'query': 'Herramienta de cálculo y usos internos franquicia', 'doc': 'ProcesoAnaliticaClientes_V2.md'})


Ver el texto completo de las fuentes citadas en la última respuesta:

In [12]:
for chunk_id in ultima.sources:
    hit = next(h for h in retriever.neighbors(chunk_id, 0))
    display(Markdown(f"---\n**`{hit.chunk_id}`** — {hit.path}\n\n{hit.text}"))

---
**`ProcesoAnaliticaClientes_V2.md#3`** — 

- Las no capturadas automáticamente deben ser solicitadas por ventas y validadas por trading.

- Las operaciones sin cliente asignado no generan franquicia; su resultado se asigna íntegramente como apalancamiento.

La información es gestionada de forma centralizada por Finanzas CIB Holding, y cualquier modificación en la franquicia se refleja automáticamente en el cálculo del apalancamiento.

**3.2 Componentes del pricing**

El precio final de una operación con cliente se construye mediante la siguiente fórmula:

*Precio final = Fair Price ± Ajustes XVA + Margen comercial*

El cálculo puede derivarse:

- Automáticamente, a través de las plataformas Murex, STAR y Calypso, por captura directa de precios o diferencias registradas ("triadas").

- Manual supervisado, cuando la operación no reside en plataforma (ej. comisiones de DCM), previa validación cruzada entre ventas y trading.

Puede consultarse el documento [<u>"Front Office Pricing Methodology 2021 March.pdf"</u>](https://drive.google.com/file/d/1cXNgKIZA0b0S_PVMTFo0_HxyIa0xX0gC/view?usp=drive_link), que explica en detalle todos los componentes que forman parte del cálculo de la fórmula anterior.

**3.3 Proceso de obtención del valor**

Una vez formalizada la operación con el cliente, el cálculo y registro de la franquicia se lleva a cabo mediante un flujo estructurado compuesto por las siguientes etapas: